# Reproducibility workflow: multi-event Sentinel-1 flood mapping

This notebook contains the analytical workflow supporting the manuscript **“Multi-Event Sentinel-1 Flood Mapping Using Adaptive Otsu Thresholding and Machine Learning in the Lokoja–Bassa–Kogi Sector of the Niger–Benue Confluence.”**

The notebook is intentionally limited to the methods and analyses reported in the manuscript. Superseded experiments, deep-learning development, runtime-recovery cells, personal cloud-storage paths, and exploratory cartography are not included. The study-area map shown in the manuscript was prepared separately in ArcGIS Pro and is not generated here.

The default workflow uses repository-relative paths. No Google Drive mount and no author-specific Earth Engine project identifier are required. Processed reproducibility files may be stored inside `data/` or supplied through the `FLOOD_DATA_ROOT` environment variable. If a public archive is deposited separately, its URL may be supplied through `FLOOD_REPRO_ARCHIVE_URL`.

Expected processed inputs are documented in the configuration cell. The NEMA flooded-location file is optional because redistribution may be restricted; the external-validation section is skipped cleanly when that file is absent.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT AND CONFIGURATION
# =============================================================================

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import shutil
import sys
import urllib.request
import warnings
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.stats import beta, binomtest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    jaccard_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterSampler, StratifiedGroupKFold

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------------------------------------------------------
# Portable repository paths
# -----------------------------------------------------------------------------

REPO_ROOT = Path(os.environ.get("FLOOD_REPO_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("FLOOD_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
RESULTS_DIR = REPO_ROOT / "results"
MODELS_DIR = REPO_ROOT / "models"
FIGURES_DIR = REPO_ROOT / "figures"

for directory in (RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Optional public data archive. Leave empty when the data folder is distributed
# with the repository. A Zenodo/OSF/Figshare archive can be supplied later.
REPRO_ARCHIVE_URL = os.environ.get("FLOOD_REPRO_ARCHIVE_URL", "").strip()

# -----------------------------------------------------------------------------
# Execution controls
# -----------------------------------------------------------------------------

RUN_SOURCE_PREPROCESSING = False   # Builds Earth Engine source images; no Drive mount.
REBUILD_LOEO_IF_POSSIBLE = False  # Uses reference_pixels.csv when available.
RUN_PRIMARY_MODEL_SELECTION = True
RUN_FULL_AREA_MAPS = True
RUN_PAIRED_INFERENCE = True
RUN_SHAP = True
RUN_SENSITIVITY = True             # Includes the 195-evaluation robustness analysis.
RUN_EXTERNAL_VALIDATION = True
VERIFY_PUBLISHED_RESULTS = True

# -----------------------------------------------------------------------------
# Locked study constants
# -----------------------------------------------------------------------------

YEARS = (2018, 2022, 2024)
FOLDS = tuple(f"LOEO_test_{year}" for year in YEARS)
ANALYSIS_CRS = "EPSG:32632"
PIXEL_SIZE_M = 10.0
PIXEL_AREA_KM2 = (PIXEL_SIZE_M ** 2) / 1_000_000.0

FEATURES = [
    "VV_pre",
    "VH_pre",
    "VV_event",
    "VH_event",
    "dVV",
    "dVH",
    "PolContrast_pre",
    "PolContrast_event",
    "Elevation",
    "Slope",
    "HAND",
    "Log_UPA",
]

DYNAMIC_FEATURES = FEATURES[:8]
STATIC_FEATURES = FEATURES[8:]

# Exact Sentinel-1 acquisitions retained in the manuscript.
S1_ACQUISITIONS = {
    2018: {
        "pass": "ASCENDING",
        "relative_orbit": 30,
        "pre_dates": ("2018-09-04", "2018-09-16"),
        "event_dates": ("2018-09-22", "2018-09-28"),
    },
    2022: {
        "pass": "ASCENDING",
        "relative_orbit": 30,
        "pre_dates": ("2022-08-26", "2022-09-07"),
        "event_dates": ("2022-09-19", "2022-10-13"),
    },
    2024: {
        "pass": "ASCENDING",
        "relative_orbit": 30,
        "pre_dates": ("2024-09-20", "2024-10-02"),
        "event_dates": ("2024-10-14", "2024-10-26"),
    },
}

MODEL_SEED = 42
BOOTSTRAP_SEED = 20260811
N_BOOTSTRAP = 5000
LGBM_THRESHOLD = 0.53
THRESHOLD_GRID = np.round(np.arange(0.25, 0.751, 0.01), 2)

OTSU_HIST_MIN_DB = -40.0
OTSU_HIST_MAX_DB = 10.0
OTSU_BIN_WIDTH_DB = 0.025
EXPECTED_OTSU_THRESHOLDS = {
    2018: -14.8125,
    2022: -13.8375,
    2024: -15.2375,
}

LOCKED_MODELS = {
    "RandomForest": {
        "params": {
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_split": 2,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
        },
        "threshold": 0.51,
        "mean_validation_iou": 0.9633445884257904,
    },
    "XGBoost": {
        "params": {
            "n_estimators": 200,
            "max_depth": 5,
            "learning_rate": 0.05,
            "subsample": 0.85,
            "colsample_bytree": 0.90,
            "min_child_weight": 5,
            "reg_alpha": 0.50,
            "reg_lambda": 5.0,
        },
        "threshold": 0.48,
        "mean_validation_iou": 0.9649171445655024,
    },
    "LightGBM": {
        "params": {
            "n_estimators": 350,
            "num_leaves": 31,
            "max_depth": -1,
            "learning_rate": 0.10,
            "min_child_samples": 20,
            "subsample": 0.85,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.50,
            "reg_lambda": 0.0,
        },
        "threshold": 0.53,
        "mean_validation_iou": 0.966045,
    },
}

EXPECTED_TEST_COUNTS = {2018: 4999, 2022: 4999, 2024: 5000}
EXPECTED_M3_CONFUSION = {
    2018: {"TN": 2447, "FP": 52, "FN": 292, "TP": 2208},
    2022: {"TN": 2439, "FP": 60, "FN": 21, "TP": 2479},
    2024: {"TN": 2410, "FP": 90, "FN": 147, "TP": 2353},
}

# -----------------------------------------------------------------------------
# Repository-relative input locations
# -----------------------------------------------------------------------------

ROI_FILE = DATA_ROOT / "inputs" / "roi.geojson"
GRID_FILE = DATA_ROOT / "processed" / "model_grid.json"
REFERENCE_PIXELS_FILE = DATA_ROOT / "processed" / "reference_pixels_12predictors.csv"
LOEO_FILE = DATA_ROOT / "processed" / "loeo_12predictor_table.csv"
NEMA_FILE = DATA_ROOT / "restricted" / "nema_flood_locations.csv"

PREDICTOR_STACKS = {
    year: DATA_ROOT / "processed" / "rasters" / f"predictor_stack_{year}_12band_10m.tif"
    for year in YEARS
}
JRC_PERMANENT_WATER = DATA_ROOT / "processed" / "rasters" / "jrc_permanent_water_10m.tif"

OTSU_MAPS = {
    year: RESULTS_DIR / f"otsu_3class_{year}.tif"
    for year in YEARS
}
LGBM_MAPS = {
    year: RESULTS_DIR / f"lightgbm_3class_{year}.tif"
    for year in YEARS
}


def _safe_extract_zip(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)


def fetch_reproducibility_archive() -> None:
    """Download an optional public reproducibility archive without Google Drive."""
    if not REPRO_ARCHIVE_URL:
        return
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = REPO_ROOT / "_reproducibility_data.zip"
    if not archive_path.exists():
        print("Downloading reproducibility archive...")
        urllib.request.urlretrieve(REPRO_ARCHIVE_URL, archive_path)
    _safe_extract_zip(archive_path, REPO_ROOT)


fetch_reproducibility_archive()

print("Repository root:", REPO_ROOT)
print("Data root:", DATA_ROOT)
print("Python:", sys.version.split()[0])
print("Predictors:", len(FEATURES))


## 2. Sentinel-1 preprocessing and acquisition matching

This section reconstructs the public-source image objects used by the analysis. It does not generate the study-area figure. The functions use the user's own Earth Engine authentication when source reconstruction is requested. The repository does not contain an author-specific Earth Engine project ID.

`COPERNICUS/S1_GRD` is used in the form supplied by Google Earth Engine. No separate radiometric terrain-flattening step is added here. Same-date tiles are mosaicked before the two selected acquisition dates in each phase are median-composited.

In [ ]:
# =============================================================================
# 2. SENTINEL-1 PREPROCESSING AND ACQUISITION MATCHING
# =============================================================================


def initialize_earth_engine():
    """Initialise Earth Engine using the current user's credentials."""
    import ee

    project = os.environ.get("EE_PROJECT", "").strip()
    try:
        if project:
            ee.Initialize(project=project)
        else:
            ee.Initialize()
    except Exception:
        ee.Authenticate()
        if project:
            ee.Initialize(project=project)
        else:
            ee.Initialize()
    return ee


def roi_to_ee_geometry(ee, vector_path: Path):
    import geopandas as gpd

    if not vector_path.exists():
        raise FileNotFoundError(
            f"ROI vector is required for source reconstruction: {vector_path}"
        )
    gdf = gpd.read_file(vector_path)
    if gdf.empty or gdf.crs is None:
        raise RuntimeError("ROI vector is empty or has no CRS.")
    gdf = gdf.to_crs("EPSG:4326")
    geojson = json.loads(gdf.to_json())
    return ee.FeatureCollection(geojson).geometry()


def build_s1_base_collection(ee, roi):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("resolution_meters", 10))
    )


def build_daily_s1_mosaic(ee, base_collection, roi, date_string):
    start = ee.Date(date_string)
    end = start.advance(1, "day")
    daily = base_collection.filterDate(start, end).select(["VV", "VH"])
    return (
        daily.mosaic()
        .clip(roi)
        .toFloat()
        .set("system:time_start", start.millis())
        .set("acquisition_date", date_string)
    )


def build_phase_composite(ee, base_collection, roi, pass_direction, relative_orbit, dates):
    matched = (
        base_collection
        .filter(ee.Filter.eq("orbitProperties_pass", pass_direction))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", int(relative_orbit)))
    )
    daily_images = [
        build_daily_s1_mosaic(ee, matched, roi, date_string)
        for date_string in dates
    ]
    return (
        ee.ImageCollection.fromImages(daily_images)
        .median()
        .select(["VV", "VH"])
        .clip(roi)
        .toFloat()
    )


def build_dynamic_predictors(ee, pre, event, roi):
    vv_pre = pre.select("VV").rename("VV_pre")
    vh_pre = pre.select("VH").rename("VH_pre")
    vv_event = event.select("VV").rename("VV_event")
    vh_event = event.select("VH").rename("VH_event")

    dvv = vv_event.subtract(vv_pre).rename("dVV")
    dvh = vh_event.subtract(vh_pre).rename("dVH")
    pc_pre = vv_pre.subtract(vh_pre).rename("PolContrast_pre")
    pc_event = vv_event.subtract(vh_event).rename("PolContrast_event")

    # Retain only pixels for which all four original SAR observations are valid.
    valid = (
        vv_pre.mask()
        .And(vh_pre.mask())
        .And(vv_event.mask())
        .And(vh_event.mask())
    )

    return (
        ee.Image.cat([
            vv_pre, vh_pre, vv_event, vh_event,
            dvv, dvh, pc_pre, pc_event,
        ])
        .updateMask(valid)
        .clip(roi)
        .toFloat()
    )


def build_source_predictor_images():
    """Return Earth Engine predictor images and the independent JRC water mask."""
    ee = initialize_earth_engine()
    roi = roi_to_ee_geometry(ee, ROI_FILE)
    s1 = build_s1_base_collection(ee, roi)

    static = ee.Image.cat([
        ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2")
        .filterBounds(roi).select("DSM").mosaic().clip(roi).rename("Elevation").toFloat(),
        ee.Terrain.slope(
            ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2")
            .filterBounds(roi).select("DSM").mosaic().clip(roi)
        ).rename("Slope").toFloat(),
        ee.Image("MERIT/Hydro/v1_0_1").select("hnd").clip(roi).rename("HAND").toFloat(),
        ee.Image("MERIT/Hydro/v1_0_1").select("upa").clip(roi)
        .max(0).add(1).log10().rename("Log_UPA").toFloat(),
    ])

    jrc_permanent = (
        ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
        .select("seasonality")
        .eq(12)
        .rename("PermanentWater")
        .clip(roi)
        .toByte()
    )

    stacks = {}
    for year, cfg in S1_ACQUISITIONS.items():
        pre = build_phase_composite(
            ee, s1, roi, cfg["pass"], cfg["relative_orbit"], cfg["pre_dates"]
        )
        event = build_phase_composite(
            ee, s1, roi, cfg["pass"], cfg["relative_orbit"], cfg["event_dates"]
        )
        dynamic = build_dynamic_predictors(ee, pre, event, roi)
        stacks[year] = ee.Image.cat([dynamic, static]).rename(FEATURES).clip(roi).toFloat()

    return stacks, jrc_permanent


if RUN_SOURCE_PREPROCESSING:
    EE_PREDICTOR_IMAGES, EE_JRC_PERMANENT_WATER = build_source_predictor_images()
    for year in YEARS:
        print(year, EE_PREDICTOR_IMAGES[year].bandNames().getInfo())
else:
    print("Source-image reconstruction is disabled. Archived processed inputs will be used.")


## 3. Predictor construction and grid harmonisation

The archived local predictor stacks used for model reproduction must contain the 12 variables in the exact order defined above and share the same 10 m EPSG:32632 grid. Continuous terrain/hydrological layers were aligned to this grid during data preparation; the JRC permanent-water reference is stored separately and is not a model predictor.

In [ ]:
# =============================================================================
# 3. PREDICTOR CONSTRUCTION AND GRID HARMONISATION
# =============================================================================


def _same_transform(a, b, atol=1e-9):
    return np.allclose(tuple(a)[:6], tuple(b)[:6], atol=atol, rtol=0)


def validate_predictor_rasters():
    import rasterio

    available = [path for path in PREDICTOR_STACKS.values() if path.exists()]
    if not available:
        print("No local predictor stacks were found; raster-dependent stages will be skipped.")
        return False

    reference_grid = None
    for year in YEARS:
        path = PREDICTOR_STACKS[year]
        if not path.exists():
            raise FileNotFoundError(path)

        with rasterio.open(path) as src:
            if src.count != 12:
                raise RuntimeError(f"{year}: expected 12 predictor bands, found {src.count}.")
            if src.crs is None or src.crs.to_string().upper() != ANALYSIS_CRS:
                raise RuntimeError(f"{year}: expected {ANALYSIS_CRS}, found {src.crs}.")
            if not np.isclose(abs(src.transform.a), PIXEL_SIZE_M):
                raise RuntimeError(f"{year}: x resolution is not 10 m.")
            if not np.isclose(abs(src.transform.e), PIXEL_SIZE_M):
                raise RuntimeError(f"{year}: y resolution is not 10 m.")

            current = (src.width, src.height, src.crs, src.transform)
            if reference_grid is None:
                reference_grid = current
            else:
                if current[:3] != reference_grid[:3] or not _same_transform(current[3], reference_grid[3]):
                    raise RuntimeError(f"{year}: predictor grid differs from the common analysis grid.")

            if src.descriptions and any(src.descriptions):
                observed = [x for x in src.descriptions]
                if observed != FEATURES:
                    raise RuntimeError(
                        f"{year}: raster band descriptions do not match the locked predictor order."
                    )

    if JRC_PERMANENT_WATER.exists():
        with rasterio.open(PREDICTOR_STACKS[2018]) as psrc, rasterio.open(JRC_PERMANENT_WATER) as wsrc:
            if (wsrc.width, wsrc.height, wsrc.crs) != (psrc.width, psrc.height, psrc.crs):
                raise RuntimeError("JRC permanent-water raster does not match the analysis grid.")
            if not _same_transform(wsrc.transform, psrc.transform):
                raise RuntimeError("JRC permanent-water raster transform differs from the analysis grid.")

    print("Predictor-grid QC: PASSED")
    return True


RASTERS_AVAILABLE = validate_predictor_rasters()


## 4. Reference-sample preparation

The workflow accepts either (a) a unique 14,998-row reference table with predictor values and projected pixel centres, from which the LOEO partition can be rebuilt, or (b) the archived expanded LOEO table. This design permits verification even when the underlying reference coordinates cannot be redistributed publicly.

In [ ]:
# =============================================================================
# 4. REFERENCE-SAMPLE PREPARATION
# =============================================================================

REFERENCE_REQUIRED = [
    "year",
    "class_id",
    "global_pixel_row",
    "global_pixel_col",
    "pixel_center_x_m",
    "pixel_center_y_m",
    *FEATURES,
]

LOEO_REQUIRED = [
    "fold",
    "split",
    "year",
    "class_id",
    "global_pixel_row",
    "global_pixel_col",
    *FEATURES,
]


def load_reference_pixels(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [c for c in REFERENCE_REQUIRED if c not in df.columns]
    if missing:
        raise RuntimeError(f"Reference table missing columns: {missing}")

    key = ["year", "global_pixel_row", "global_pixel_col"]
    df = df.drop_duplicates(subset=key, keep="first").copy()
    if len(df) != 14_998:
        raise RuntimeError(f"Expected 14,998 unique reference pixels; found {len(df):,}.")
    if set(df["year"].astype(int).unique()) != set(YEARS):
        raise RuntimeError("Reference table contains unexpected event years.")
    if set(df["class_id"].astype(int).unique()) != {0, 1}:
        raise RuntimeError("Reference labels must be binary (0/1).")
    if not np.isfinite(df[FEATURES].to_numpy(float)).all():
        raise RuntimeError("Reference table contains non-finite predictor values.")
    return df


def load_archived_loeo(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [c for c in LOEO_REQUIRED if c not in df.columns]
    if missing:
        raise RuntimeError(f"LOEO table missing columns: {missing}")
    if set(df["fold"].unique()) != set(FOLDS):
        raise RuntimeError("LOEO fold names differ from the locked design.")
    if not set(df["split"].unique()).issubset({"train", "validation", "test"}):
        raise RuntimeError("Unexpected LOEO split label.")
    if not np.isfinite(df[FEATURES].to_numpy(float)).all():
        raise RuntimeError("LOEO table contains non-finite predictor values.")
    return df


reference_pixels = None
loeo = None

if REFERENCE_PIXELS_FILE.exists():
    reference_pixels = load_reference_pixels(REFERENCE_PIXELS_FILE)
    print("Unique reference pixels:", f"{len(reference_pixels):,}")

if LOEO_FILE.exists() and not REBUILD_LOEO_IF_POSSIBLE:
    loeo = load_archived_loeo(LOEO_FILE)
    print("Archived LOEO rows:", f"{len(loeo):,}")


## 5. LOEO partitioning and spatial blocking

For each outer fold, one complete flood event is withheld as the test event. The two remaining events form the development pool. Development samples are assigned to 10 km spatial blocks and split with five-fold stratified group cross-validation. The validation fold whose class/event composition is closest to the development pool is retained. Training centres within 2.56 km Chebyshev distance of any validation centre are removed to prevent overlap of the 2.56 km square analysis neighbourhoods used during the original partition construction.

In [ ]:
# =============================================================================
# 5. LOEO PARTITIONING AND SPATIAL BLOCKING
# =============================================================================

SPATIAL_BLOCK_SIZE_M = 10_000.0
N_SPATIAL_FOLDS = 5
SPLIT_RANDOM_STATE = 20260807
PATCH_NONOVERLAP_GUARD_M = 2560.0


def _load_grid_origin():
    if GRID_FILE.exists():
        cfg = json.loads(GRID_FILE.read_text(encoding="utf-8"))
        xmin = float(cfg.get("xmin", cfg.get("x_min", np.nan)))
        ymax = float(cfg.get("ymax", cfg.get("y_max", np.nan)))
        if np.isfinite(xmin) and np.isfinite(ymax):
            return xmin, ymax
    if reference_pixels is not None:
        # Grid-aligned fallback; only used when the archived model-grid JSON is absent.
        xmin = math.floor(reference_pixels["pixel_center_x_m"].min() / PIXEL_SIZE_M) * PIXEL_SIZE_M
        ymax = math.ceil(reference_pixels["pixel_center_y_m"].max() / PIXEL_SIZE_M) * PIXEL_SIZE_M
        return xmin, ymax
    raise FileNotFoundError("Model-grid origin cannot be determined.")


def build_loeo_partitions(reference_df: pd.DataFrame):
    grid_xmin, grid_ymax = _load_grid_origin()
    labels = reference_df.copy()

    labels["block_col"] = np.floor(
        (labels["pixel_center_x_m"] - grid_xmin) / SPATIAL_BLOCK_SIZE_M
    ).astype(int)
    labels["block_row"] = np.floor(
        (grid_ymax - labels["pixel_center_y_m"]) / SPATIAL_BLOCK_SIZE_M
    ).astype(int)
    labels["block_id"] = (
        "R" + labels["block_row"].astype(str) + "_C" + labels["block_col"].astype(str)
    )
    labels["year_class"] = labels["year"].astype(str) + "_" + labels["class_id"].astype(str)

    all_usable = []
    qc_rows = []

    for test_year in YEARS:
        test_df = labels.loc[labels["year"] == test_year].copy()
        development = labels.loc[labels["year"] != test_year].copy()

        splitter = StratifiedGroupKFold(
            n_splits=N_SPATIAL_FOLDS,
            shuffle=True,
            random_state=SPLIT_RANDOM_STATE,
        )

        dev_prop = development["year_class"].value_counts(normalize=True)
        candidates = []
        for fold_number, (train_idx, val_idx) in enumerate(
            splitter.split(development, y=development["year_class"], groups=development["block_id"])
        ):
            val_candidate = development.iloc[val_idx]
            val_prop = val_candidate["year_class"].value_counts(normalize=True)
            categories = sorted(set(dev_prop.index) | set(val_prop.index))
            distribution_error = sum(
                abs(float(dev_prop.get(k, 0)) - float(val_prop.get(k, 0)))
                for k in categories
            )
            fraction_error = abs(len(val_candidate) / len(development) - 1.0 / N_SPATIAL_FOLDS)
            candidates.append((distribution_error + fraction_error, fold_number, train_idx, val_idx))

        _, selected_number, train_idx, val_idx = min(candidates, key=lambda x: x[0])
        train_candidate = development.iloc[train_idx].copy()
        validation = development.iloc[val_idx].copy()

        val_xy = validation[["pixel_center_x_m", "pixel_center_y_m"]].to_numpy(float)
        train_xy = train_candidate[["pixel_center_x_m", "pixel_center_y_m"]].to_numpy(float)
        tree = cKDTree(val_xy)
        nearest, _ = tree.query(train_xy, k=1, p=np.inf, workers=-1)
        guard = nearest < PATCH_NONOVERLAP_GUARD_M

        train = train_candidate.loc[~guard].copy()
        guard_excluded = train_candidate.loc[guard].copy()
        train["nearest_validation_center_m"] = nearest[~guard]

        train["split"] = "train"
        validation["split"] = "validation"
        test_df["split"] = "test"
        fold_name = f"LOEO_test_{test_year}"
        for part in (train, validation, test_df):
            part["fold"] = fold_name
            part["outer_test_year"] = test_year

        usable = pd.concat([train, validation, test_df], ignore_index=True)
        all_usable.append(usable)

        min_separation = float(train["nearest_validation_center_m"].min()) if len(train) else np.nan
        qc_rows.append({
            "fold": fold_name,
            "selected_spatial_validation_fold": selected_number,
            "train_n": len(train),
            "validation_n": len(validation),
            "test_n": len(test_df),
            "guard_excluded_n": len(guard_excluded),
            "minimum_train_validation_chebyshev_m": min_separation,
            "required_nonoverlap_distance_m": PATCH_NONOVERLAP_GUARD_M,
            "fold_qc": bool(
                (np.isnan(min_separation) or min_separation >= PATCH_NONOVERLAP_GUARD_M)
                and (train["year"] != test_year).all()
                and (validation["year"] != test_year).all()
                and (test_df["year"] == test_year).all()
            ),
        })

    return pd.concat(all_usable, ignore_index=True), pd.DataFrame(qc_rows)


if loeo is None:
    if reference_pixels is None:
        raise FileNotFoundError(
            "Provide either data/processed/reference_pixels_12predictors.csv "
            "or data/processed/loeo_12predictor_table.csv."
        )
    loeo, loeo_qc = build_loeo_partitions(reference_pixels)
    LOEO_FILE.parent.mkdir(parents=True, exist_ok=True)
    loeo.to_csv(LOEO_FILE, index=False)
    loeo_qc.to_csv(RESULTS_DIR / "loeo_partition_qc.csv", index=False)
else:
    loeo_qc = None

# Core fold checks.
for year in YEARS:
    fold = f"LOEO_test_{year}"
    test = loeo[(loeo["fold"] == fold) & (loeo["split"] == "test")]
    if len(test) != EXPECTED_TEST_COUNTS[year]:
        raise RuntimeError(f"{year}: expected {EXPECTED_TEST_COUNTS[year]} test rows, found {len(test)}.")
    if not (test["year"].astype(int) == year).all():
        raise RuntimeError(f"{year}: held-out test fold contains another event year.")

print(loeo.groupby(["fold", "split"]).size())
print("Usable expanded LOEO rows:", f"{len(loeo):,}")


## 6. Event-specific adaptive Otsu benchmark

Otsu thresholds are estimated independently from the full valid `VV_event` distribution for each event. The quality-control histogram domain is −40 to +10 dB with 0.025 dB bins. Reference labels, machine-learning predictions, and the permanent-water mask are not used to select the threshold. JRC seasonality = 12 is applied only afterwards to separate permanent water from temporary inundation.

In [ ]:
# =============================================================================
# 6. EVENT-SPECIFIC ADAPTIVE OTSU BENCHMARK
# =============================================================================


def calculate_otsu_from_histogram(histogram: np.ndarray, bin_centers: np.ndarray):
    histogram = np.asarray(histogram, dtype=np.float64)
    if histogram.sum() <= 0:
        raise RuntimeError("Histogram contains no valid pixels.")

    probability = histogram / histogram.sum()
    omega = np.cumsum(probability)
    cumulative_mean = np.cumsum(probability * bin_centers)
    global_mean = cumulative_mean[-1]
    denominator = omega * (1.0 - omega)

    between = np.full(histogram.size, np.nan, dtype=float)
    valid = denominator > 0
    between[valid] = (
        (global_mean * omega[valid] - cumulative_mean[valid]) ** 2
        / denominator[valid]
    )
    index = int(np.nanargmax(between))
    return float(bin_centers[index]), between


def otsu_threshold_from_raster(path: Path, band_index: int = 3):
    import rasterio

    edges = np.arange(
        OTSU_HIST_MIN_DB,
        OTSU_HIST_MAX_DB + OTSU_BIN_WIDTH_DB,
        OTSU_BIN_WIDTH_DB,
        dtype=np.float64,
    )
    centers = (edges[:-1] + edges[1:]) / 2.0
    histogram = np.zeros(len(centers), dtype=np.int64)

    with rasterio.open(path) as src:
        for _, window in src.block_windows(band_index):
            arr = src.read(band_index, window=window, masked=True).astype(np.float64)
            values = arr.compressed()
            values = values[np.isfinite(values)]
            values = values[(values >= OTSU_HIST_MIN_DB) & (values <= OTSU_HIST_MAX_DB)]
            if values.size:
                histogram += np.histogram(values, bins=edges)[0]

    threshold, between = calculate_otsu_from_histogram(histogram, centers)
    return threshold, histogram, centers, between


def write_otsu_map(predictor_path: Path, permanent_water_path: Path, output_path: Path, threshold: float):
    import rasterio

    with rasterio.open(predictor_path) as psrc, rasterio.open(permanent_water_path) as wsrc:
        profile = psrc.profile.copy()
        profile.update(count=1, dtype="uint8", nodata=255, compress="DEFLATE")
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(output_path, "w", **profile) as dst:
            for _, window in psrc.block_windows(3):
                vv = psrc.read(3, window=window, masked=True).astype(np.float32)
                water = wsrc.read(1, window=window, masked=True)
                valid = (~np.ma.getmaskarray(vv)) & np.isfinite(vv.filled(np.nan))
                valid &= ~np.ma.getmaskarray(water)

                out = np.full(vv.shape, 255, dtype=np.uint8)
                permanent = valid & (water.filled(0) == 1)
                event_water = valid & (vv.filled(np.nan) <= threshold)
                flood = event_water & ~permanent
                nonflood = valid & ~permanent & ~flood

                out[nonflood] = 0
                out[permanent] = 1
                out[flood] = 2
                dst.write(out, 1, window=window)


otsu_thresholds = {}
otsu_histograms = {}

if RASTERS_AVAILABLE:
    for year in YEARS:
        threshold, hist, centers, between = otsu_threshold_from_raster(PREDICTOR_STACKS[year])
        otsu_thresholds[year] = threshold
        otsu_histograms[year] = (hist, centers, between)
        print(f"{year}: Otsu threshold = {threshold:.4f} dB")

        if VERIFY_PUBLISHED_RESULTS and not np.isclose(
            threshold, EXPECTED_OTSU_THRESHOLDS[year], atol=OTSU_BIN_WIDTH_DB / 2 + 1e-12
        ):
            raise RuntimeError(
                f"{year}: reproduced Otsu threshold {threshold:.4f} differs from the archived result "
                f"{EXPECTED_OTSU_THRESHOLDS[year]:.4f}."
            )

        if JRC_PERMANENT_WATER.exists() and RUN_FULL_AREA_MAPS:
            write_otsu_map(
                PREDICTOR_STACKS[year], JRC_PERMANENT_WATER, OTSU_MAPS[year], threshold
            )

    pd.DataFrame([
        {"Year": year, "Otsu_Threshold_dB": otsu_thresholds[year]}
        for year in YEARS
    ]).to_csv(RESULTS_DIR / "otsu_thresholds.csv", index=False)

    # Reproducibility histogram; manuscript cartographic styling is kept separate.
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False)
        for ax, year in zip(axes, YEARS):
            hist, centers, _ = otsu_histograms[year]
            ax.plot(centers, hist)
            ax.axvline(otsu_thresholds[year], linestyle="--")
            ax.set_title(str(year))
            ax.set_xlabel("VV event backscatter (dB)")
            ax.set_ylabel("Pixel count")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "otsu_threshold_histograms.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
    except Exception as exc:
        print("Histogram figure skipped:", exc)
else:
    # Published thresholds remain recorded for downstream table reproduction when
    # the large predictor rasters are stored in a separate archive.
    otsu_thresholds = EXPECTED_OTSU_THRESHOLDS.copy()
    print("Predictor rasters unavailable; using archived Otsu thresholds for non-raster stages.")


## 7. Machine-learning development and model selection

Random Forest, XGBoost, and LightGBM are compared on development data only. Each model family uses one prespecified baseline plus five `ParameterSampler` configurations. Flood IoU at probability 0.50 is the primary candidate-selection metric. After the winning family is selected, one common probability threshold is chosen from 0.25–0.75 using mean validation Flood IoU. Held-out event rows are never used in this stage.

In [ ]:
# =============================================================================
# 7. MACHINE-LEARNING DEVELOPMENT AND MODEL SELECTION
# =============================================================================

RF_BASELINE = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
}
RF_SPACE = {
    "n_estimators": [200, 250, 300, 350, 400],
    "max_depth": [None, 10, 16, 24],
    "min_samples_split": [2, 4, 8],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.6, 0.8, 1.0],
}

XGB_BASELINE = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 1,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}
XGB_SPACE = {
    "n_estimators": [200, 250, 300, 350, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.03, 0.05, 0.08, 0.10],
    "subsample": [0.75, 0.85, 0.90, 1.0],
    "colsample_bytree": [0.70, 0.80, 0.90, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_alpha": [0.0, 0.10, 0.50],
    "reg_lambda": [1.0, 2.0, 5.0],
}

LGB_BASELINE = {
    "n_estimators": 300,
    "num_leaves": 31,
    "max_depth": -1,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
}
LGB_SPACE = {
    "n_estimators": [200, 250, 300, 350, 400],
    "num_leaves": [15, 31, 47, 63],
    "max_depth": [-1, 6, 10, 14],
    "learning_rate": [0.03, 0.05, 0.08, 0.10],
    "min_child_samples": [10, 20, 30, 40],
    "subsample": [0.75, 0.85, 0.90, 1.0],
    "colsample_bytree": [0.70, 0.80, 0.90, 1.0],
    "reg_alpha": [0.0, 0.10, 0.50],
    "reg_lambda": [0.0, 1.0, 3.0],
}

BASELINES = {"RandomForest": RF_BASELINE, "XGBoost": XGB_BASELINE, "LightGBM": LGB_BASELINE}
SPACES = {"RandomForest": RF_SPACE, "XGBoost": XGB_SPACE, "LightGBM": LGB_SPACE}


def build_primary_model(family: str, params: dict, seed: int = MODEL_SEED):
    n_jobs = min(2, os.cpu_count() or 2)
    if family == "RandomForest":
        return RandomForestClassifier(
            **params,
            class_weight="balanced_subsample",
            random_state=seed,
            n_jobs=n_jobs,
        )
    if family == "XGBoost":
        return XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=n_jobs,
            verbosity=0,
        )
    if family == "LightGBM":
        return LGBMClassifier(
            **params,
            objective="binary",
            random_state=seed,
            n_jobs=n_jobs,
            verbosity=-1,
            force_col_wise=True,
            subsample_freq=1,
        )
    raise ValueError(family)


def binary_metrics(y_true, y_pred, probability=None):
    y_true = np.asarray(y_true, dtype=np.uint8)
    y_pred = np.asarray(y_pred, dtype=np.uint8)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "IoU": jaccard_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    if probability is not None:
        out["ROC_AUC"] = roc_auc_score(y_true, probability)
    return out


def primary_candidate_sets():
    return {
        "RandomForest": [RF_BASELINE, *list(ParameterSampler(RF_SPACE, n_iter=5, random_state=MODEL_SEED + 1))],
        "XGBoost": [XGB_BASELINE, *list(ParameterSampler(XGB_SPACE, n_iter=5, random_state=MODEL_SEED + 2))],
        "LightGBM": [LGB_BASELINE, *list(ParameterSampler(LGB_SPACE, n_iter=5, random_state=MODEL_SEED + 3))],
    }


selection_summary = None
winner_validation = None
selected_family = "LightGBM"
selected_params = LOCKED_MODELS["LightGBM"]["params"].copy()
selected_threshold = LGBM_THRESHOLD

if RUN_PRIMARY_MODEL_SELECTION:
    candidate_sets = primary_candidate_sets()
    fit_rows = []

    for family, candidates in candidate_sets.items():
        for candidate_id, params in enumerate(candidates):
            for fold in FOLDS:
                train = loeo[(loeo["fold"] == fold) & (loeo["split"] == "train")]
                val = loeo[(loeo["fold"] == fold) & (loeo["split"] == "validation")]

                model = build_primary_model(family, dict(params))
                model.fit(train[FEATURES].to_numpy(np.float32), train["class_id"].to_numpy(np.uint8))
                probability = model.predict_proba(val[FEATURES].to_numpy(np.float32))[:, 1]
                prediction = (probability >= 0.50).astype(np.uint8)
                metric = binary_metrics(val["class_id"].to_numpy(np.uint8), prediction, probability)
                fit_rows.append({
                    "Model": family,
                    "Candidate_ID": candidate_id,
                    "Fold": fold,
                    "Validation_N": len(val),
                    "Parameters": json.dumps(dict(params), sort_keys=True, default=str),
                    **metric,
                })
                del model
                gc.collect()

    fit_table = pd.DataFrame(fit_rows)
    fit_table.to_csv(RESULTS_DIR / "model_selection_all_fits.csv", index=False)

    candidate_summary = (
        fit_table.groupby(["Model", "Candidate_ID"], as_index=False)
        .agg(
            Folds=("Fold", "nunique"),
            Mean_Val_IoU=("IoU", "mean"),
            SD_Val_IoU=("IoU", "std"),
            Mean_Val_F1=("F1", "mean"),
            Mean_Val_Precision=("Precision", "mean"),
            Mean_Val_Recall=("Recall", "mean"),
            Mean_Val_Accuracy=("Accuracy", "mean"),
            Mean_Val_ROC_AUC=("ROC_AUC", "mean"),
        )
    )
    candidate_summary = candidate_summary[candidate_summary["Folds"] == 3].copy()
    best_per_model = (
        candidate_summary.sort_values(
            ["Model", "Mean_Val_IoU", "Mean_Val_F1", "Mean_Val_ROC_AUC"],
            ascending=[True, False, False, False],
        )
        .groupby("Model", as_index=False).first()
        .sort_values(["Mean_Val_IoU", "Mean_Val_F1", "Mean_Val_ROC_AUC"], ascending=False)
        .reset_index(drop=True)
    )
    selection_summary = best_per_model
    selection_summary.to_csv(RESULTS_DIR / "model_selection_summary.csv", index=False)

    winner = best_per_model.iloc[0]
    selected_family = str(winner["Model"])
    selected_id = int(winner["Candidate_ID"])
    selected_params = dict(candidate_sets[selected_family][selected_id])

    if VERIFY_PUBLISHED_RESULTS:
        if selected_family != "LightGBM":
            raise RuntimeError(f"Expected LightGBM to be selected; reproduced winner is {selected_family}.")
        locked = LOCKED_MODELS["LightGBM"]["params"]
        if json.dumps(selected_params, sort_keys=True, default=str) != json.dumps(locked, sort_keys=True, default=str):
            raise RuntimeError("Reproduced LightGBM parameters differ from the locked manuscript configuration.")

    validation_frames = []
    for fold in FOLDS:
        train = loeo[(loeo["fold"] == fold) & (loeo["split"] == "train")]
        val = loeo[(loeo["fold"] == fold) & (loeo["split"] == "validation")]
        model = build_primary_model(selected_family, selected_params)
        model.fit(train[FEATURES].to_numpy(np.float32), train["class_id"].to_numpy(np.uint8))
        p = model.predict_proba(val[FEATURES].to_numpy(np.float32))[:, 1]
        validation_frames.append(pd.DataFrame({"Fold": fold, "True_Class": val["class_id"].to_numpy(int), "Probability": p}))
        del model
        gc.collect()

    winner_validation = pd.concat(validation_frames, ignore_index=True)
    threshold_rows = []
    for threshold in THRESHOLD_GRID:
        fold_ious, fold_f1 = [], []
        for fold in FOLDS:
            d = winner_validation[winner_validation["Fold"] == fold]
            pred = (d["Probability"].to_numpy(float) >= threshold).astype(np.uint8)
            y = d["True_Class"].to_numpy(np.uint8)
            fold_ious.append(jaccard_score(y, pred, pos_label=1, zero_division=0))
            fold_f1.append(f1_score(y, pred, pos_label=1, zero_division=0))
        threshold_rows.append({"Threshold": threshold, "Mean_IoU": np.mean(fold_ious), "Mean_F1": np.mean(fold_f1)})

    threshold_scan = pd.DataFrame(threshold_rows).sort_values(["Mean_IoU", "Mean_F1"], ascending=False).reset_index(drop=True)
    selected_threshold = float(threshold_scan.iloc[0]["Threshold"])
    threshold_scan.to_csv(RESULTS_DIR / "lightgbm_validation_threshold_scan.csv", index=False)

    if VERIFY_PUBLISHED_RESULTS and not np.isclose(selected_threshold, LGBM_THRESHOLD):
        raise RuntimeError(f"Expected threshold 0.53; reproduced threshold is {selected_threshold:.2f}.")

print("Selected model:", selected_family)
print("Selected threshold:", selected_threshold)
if selection_summary is not None:
    print(selection_summary[["Model", "Mean_Val_IoU", "Mean_Val_F1", "Mean_Val_ROC_AUC"]])


## 8. Independent held-out evaluation and full-area LightGBM inference

After model family, hyperparameters, and the 0.53 decision threshold are frozen, the selected LightGBM is refitted on the training and validation portions of each development pool and applied once to the untouched event. The saved fold-specific estimators are then used for full-area inference when the archived predictor rasters are available.

In [ ]:
# =============================================================================
# 8. INDEPENDENT HELD-OUT EVALUATION AND FULL-AREA LIGHTGBM INFERENCE
# =============================================================================


def build_locked_lgbm():
    return LGBMClassifier(
        **LOCKED_MODELS["LightGBM"]["params"],
        objective="binary",
        random_state=MODEL_SEED,
        n_jobs=min(2, os.cpu_count() or 2),
        verbosity=-1,
        force_col_wise=True,
        subsample_freq=1,
    )


heldout_rows = []
heldout_frames = []
fold_models = {}

for year in YEARS:
    fold = f"LOEO_test_{year}"
    fit_df = loeo[(loeo["fold"] == fold) & (loeo["split"].isin(["train", "validation"]))].copy()
    test_df = loeo[(loeo["fold"] == fold) & (loeo["split"] == "test")].copy()

    if (fit_df["year"].astype(int) == year).any():
        raise RuntimeError(f"{year}: held-out year leaked into model fitting data.")

    model = build_locked_lgbm()
    model.fit(fit_df[FEATURES].to_numpy(np.float32), fit_df["class_id"].to_numpy(np.uint8))
    probability = model.predict_proba(test_df[FEATURES].to_numpy(np.float32))[:, 1]
    prediction = (probability >= LGBM_THRESHOLD).astype(np.uint8)
    metric = binary_metrics(test_df["class_id"].to_numpy(np.uint8), prediction, probability)

    heldout_rows.append({
        "Year": year,
        "Fit_N": len(fit_df),
        "Test_N": len(test_df),
        "Threshold": LGBM_THRESHOLD,
        **metric,
    })

    meta_cols = [
        c for c in [
            "fold", "year", "class_id", "global_pixel_row", "global_pixel_col",
            "label_uid", "block_id"
        ] if c in test_df.columns
    ]
    out = test_df[meta_cols].copy().reset_index(drop=True)
    out["Flood_Probability"] = probability
    out["LightGBM"] = prediction
    heldout_frames.append(out)

    model_path = MODELS_DIR / f"lightgbm_loeo_test_{year}.joblib"
    joblib.dump(model, model_path, compress=3)
    fold_models[year] = model

    if VERIFY_PUBLISHED_RESULTS:
        expected = EXPECTED_M3_CONFUSION[year]
        observed = {k: metric[k] for k in ("TN", "FP", "FN", "TP")}
        if observed != expected:
            raise RuntimeError(f"{year}: held-out confusion matrix changed: {observed} != {expected}")

heldout_metrics = pd.DataFrame(heldout_rows)
heldout_predictions = pd.concat(heldout_frames, ignore_index=True)
heldout_metrics.to_csv(RESULTS_DIR / "lightgbm_heldout_metrics.csv", index=False)
heldout_predictions.to_csv(RESULTS_DIR / "lightgbm_heldout_predictions.csv", index=False)

pooled_metric = binary_metrics(
    heldout_predictions["class_id"].to_numpy(np.uint8),
    heldout_predictions["LightGBM"].to_numpy(np.uint8),
    heldout_predictions["Flood_Probability"].to_numpy(float),
)
pd.DataFrame([{"Scope": "Pooled", "N": len(heldout_predictions), **pooled_metric}]).to_csv(
    RESULTS_DIR / "lightgbm_pooled_metrics.csv", index=False
)

print(heldout_metrics[["Year", "IoU", "F1", "Precision", "Recall", "Specificity", "ROC_AUC"]])
print("Pooled Flood IoU:", round(pooled_metric["IoU"], 6))


def write_lightgbm_map(predictor_path: Path, permanent_water_path: Path, output_path: Path, model):
    import rasterio

    with rasterio.open(predictor_path) as psrc, rasterio.open(permanent_water_path) as wsrc:
        profile = psrc.profile.copy()
        profile.update(count=1, dtype="uint8", nodata=255, compress="DEFLATE")
        with rasterio.open(output_path, "w", **profile) as dst:
            for _, window in psrc.block_windows(1):
                block = psrc.read(indexes=list(range(1, 13)), window=window, masked=True).astype(np.float32)
                water = wsrc.read(1, window=window, masked=True)

                masks = np.ma.getmaskarray(block)
                valid = ~masks.any(axis=0)
                valid &= np.isfinite(block.filled(np.nan)).all(axis=0)
                valid &= ~np.ma.getmaskarray(water)

                out = np.full(valid.shape, 255, dtype=np.uint8)
                if valid.any():
                    X = np.moveaxis(block.filled(np.nan), 0, -1)[valid]
                    p = model.predict_proba(X)[:, 1]
                    flood = np.zeros(valid.shape, dtype=bool)
                    flood[valid] = p >= LGBM_THRESHOLD
                    permanent = valid & (water.filled(0) == 1)
                    flood &= ~permanent
                    nonflood = valid & ~permanent & ~flood
                    out[nonflood] = 0
                    out[permanent] = 1
                    out[flood] = 2
                dst.write(out, 1, window=window)


if RUN_FULL_AREA_MAPS and RASTERS_AVAILABLE and JRC_PERMANENT_WATER.exists():
    for year in YEARS:
        write_lightgbm_map(PREDICTOR_STACKS[year], JRC_PERMANENT_WATER, LGBM_MAPS[year], fold_models[year])
        print("Saved:", LGBM_MAPS[year])
else:
    print("Full-area LightGBM raster generation skipped because required local rasters are unavailable or disabled.")

# Reproducibility ROC figure.
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.2, 4.5))
    for year in YEARS:
        d = heldout_predictions[heldout_predictions["year"].astype(int) == year]
        fpr, tpr, _ = roc_curve(d["class_id"], d["Flood_Probability"])
        auc = roc_auc_score(d["class_id"], d["Flood_Probability"])
        ax.plot(fpr, tpr, label=f"{year} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False-positive rate")
    ax.set_ylabel("True-positive rate")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "lightgbm_heldout_roc.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
except Exception as exc:
    print("ROC figure skipped:", exc)


## 9. Paired Otsu–LightGBM statistical comparison

The two methods are paired by exact event year, global raster row, and global raster column. The primary effect size is the difference in Flood IoU. Uncertainty is estimated with 5,000 paired stratified bootstrap replicates. The pooled bootstrap preserves year × class strata; event-specific bootstraps preserve class strata. Exact two-sided McNemar tests compare paired correctness, with Holm adjustment applied only to the three event-specific tests.

In [ ]:
# =============================================================================
# 9. PAIRED STATISTICAL COMPARISON
# =============================================================================


def sample_threeclass_at_reference(map_path: Path, df: pd.DataFrame, flood_class: int = 2):
    import rasterio

    rows = df["global_pixel_row"].to_numpy(np.int64)
    cols = df["global_pixel_col"].to_numpy(np.int64)
    with rasterio.open(map_path) as src:
        if (rows < 0).any() or (cols < 0).any() or (rows >= src.height).any() or (cols >= src.width).any():
            raise RuntimeError(f"Reference pixels fall outside {map_path.name}.")
        xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset="center")
        values = np.fromiter((x[0] for x in src.sample(zip(xs, ys), indexes=1)), dtype=np.int16, count=len(rows))
    if (values == 255).any():
        raise RuntimeError(f"{map_path.name}: NoData encountered at held-out reference pixels.")
    return (values == flood_class).astype(np.uint8), values


paired = None
if all(path.exists() for path in OTSU_MAPS.values()):
    otsu_frames = []
    for year in YEARS:
        d = heldout_predictions[heldout_predictions["year"].astype(int) == year].copy().reset_index(drop=True)
        pred, map_class = sample_threeclass_at_reference(OTSU_MAPS[year], d, flood_class=2)
        d["Otsu"] = pred
        d["Otsu_Map_Class"] = map_class
        otsu_frames.append(d)
    paired = pd.concat(otsu_frames, ignore_index=True)
else:
    archived = DATA_ROOT / "processed" / "otsu_heldout_predictions.csv"
    if archived.exists():
        otsu = pd.read_csv(archived)
        key = ["year", "global_pixel_row", "global_pixel_col"]
        needed = [*key, "Otsu"]
        missing = [c for c in needed if c not in otsu.columns]
        if missing:
            raise RuntimeError(f"Archived Otsu prediction table missing: {missing}")
        paired = heldout_predictions.merge(otsu[needed], on=key, how="inner", validate="one_to_one")


def flood_iou_fast(y, pred):
    y = np.asarray(y, dtype=np.uint8)
    pred = np.asarray(pred, dtype=np.uint8)
    tp = np.sum((y == 1) & (pred == 1))
    fp = np.sum((y == 0) & (pred == 1))
    fn = np.sum((y == 1) & (pred == 0))
    den = tp + fp + fn
    return float(tp / den) if den else np.nan


def paired_bootstrap(d: pd.DataFrame, pooled: bool, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    group_cols = ["year", "class_id"] if pooled else ["class_id"]
    strata = [np.asarray(ix, dtype=int) for ix in d.groupby(group_cols, sort=True).indices.values()]
    y = d["class_id"].to_numpy(np.uint8)
    o = d["Otsu"].to_numpy(np.uint8)
    l = d["LightGBM"].to_numpy(np.uint8)
    delta = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        sampled = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in strata])
        delta[b] = flood_iou_fast(y[sampled], l[sampled]) - flood_iou_fast(y[sampled], o[sampled])

    return {
        "Delta_IoU": flood_iou_fast(y, l) - flood_iou_fast(y, o),
        "CI95_Lower": float(np.percentile(delta, 2.5)),
        "CI95_Upper": float(np.percentile(delta, 97.5)),
    }


def exact_mcnemar(y, a, b):
    y = np.asarray(y)
    ca = np.asarray(a) == y
    cb = np.asarray(b) == y
    a_only = int(np.sum(ca & ~cb))
    b_only = int(np.sum(~ca & cb))
    n = a_only + b_only
    p = 1.0 if n == 0 else float(binomtest(b_only, n=n, p=0.5, alternative="two-sided").pvalue)
    return a_only, b_only, p


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    adjusted = np.empty_like(pvalues)
    running = 0.0
    m = len(pvalues)
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * pvalues[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted


if RUN_PAIRED_INFERENCE and paired is not None:
    if len(paired) != 14_998:
        raise RuntimeError(f"Expected 14,998 paired held-out observations; found {len(paired):,}.")

    comparison_rows = []
    bootstrap_rows = []
    mcnemar_rows = []
    scopes = [2018, 2022, 2024, "Pooled"]

    for scope in scopes:
        d = paired if scope == "Pooled" else paired[paired["year"].astype(int) == int(scope)]
        y = d["class_id"].to_numpy(np.uint8)
        o = d["Otsu"].to_numpy(np.uint8)
        l = d["LightGBM"].to_numpy(np.uint8)

        om = binary_metrics(y, o)
        lm = binary_metrics(y, l)
        comparison_rows.append({
            "Scope": scope,
            "N": len(d),
            "Otsu_IoU": om["IoU"],
            "LightGBM_IoU": lm["IoU"],
            "Delta_IoU": lm["IoU"] - om["IoU"],
            "Otsu_F1": om["F1"],
            "LightGBM_F1": lm["F1"],
            "Otsu_Precision": om["Precision"],
            "LightGBM_Precision": lm["Precision"],
            "Otsu_Recall": om["Recall"],
            "LightGBM_Recall": lm["Recall"],
        })

        boot = paired_bootstrap(d.reset_index(drop=True), pooled=(scope == "Pooled"))
        bootstrap_rows.append({"Scope": scope, **boot})
        o_only, l_only, p = exact_mcnemar(y, o, l)
        mcnemar_rows.append({
            "Scope": scope,
            "Otsu_correct_only": o_only,
            "LightGBM_correct_only": l_only,
            "McNemar_exact_p": p,
        })

    comparison = pd.DataFrame(comparison_rows)
    bootstrap = pd.DataFrame(bootstrap_rows)
    mcnemar = pd.DataFrame(mcnemar_rows)

    event_mask = mcnemar["Scope"].astype(str) != "Pooled"
    mcnemar.loc[event_mask, "Holm_adjusted_p"] = holm_adjust(mcnemar.loc[event_mask, "McNemar_exact_p"].to_numpy(float))

    comparison.to_csv(RESULTS_DIR / "otsu_lightgbm_paired_metrics.csv", index=False)
    bootstrap.to_csv(RESULTS_DIR / "otsu_lightgbm_bootstrap_iou.csv", index=False)
    mcnemar.to_csv(RESULTS_DIR / "otsu_lightgbm_mcnemar.csv", index=False)

    pooled_cmp = comparison.loc[comparison["Scope"] == "Pooled"].iloc[0]
    pooled_boot = bootstrap.loc[bootstrap["Scope"] == "Pooled"].iloc[0]
    print(comparison)
    print(bootstrap)
    print(mcnemar)

    if VERIFY_PUBLISHED_RESULTS:
        if not np.isclose(pooled_cmp["Otsu_IoU"], 0.875, atol=0.001):
            raise RuntimeError("Pooled Otsu IoU differs from the manuscript value.")
        if not np.isclose(pooled_cmp["LightGBM_IoU"], 0.914, atol=0.001):
            raise RuntimeError("Pooled LightGBM IoU differs from the manuscript value.")
        if not np.isclose(pooled_cmp["Delta_IoU"], 0.039, atol=0.001):
            raise RuntimeError("Pooled Delta IoU differs from the manuscript value.")
        if not (0.030 <= pooled_boot["CI95_Lower"] <= 0.032 and 0.045 <= pooled_boot["CI95_Upper"] <= 0.047):
            raise RuntimeError("Pooled bootstrap CI differs materially from the manuscript value.")
else:
    print("Paired Otsu–LightGBM inference skipped because final Otsu predictions/maps are unavailable or disabled.")


## 10. Full-area flood mapping and spatial agreement

Spatial agreement is evaluated only where both final three-class products are valid. Classes are 0 = non-flooded land, 1 = long-term permanent water, 2 = event flood, and 255 = NoData. Flood-footprint IoU and Dice are kept separate from reference-sample accuracy statistics.

In [ ]:
# =============================================================================
# 10. FULL-AREA FLOOD MAPPING AND SPATIAL AGREEMENT
# =============================================================================


def map_area_statistics(path: Path):
    import rasterio

    counts = {0: 0, 1: 0, 2: 0, 255: 0}
    with rasterio.open(path) as src:
        pixel_area_km2 = abs(src.transform.a * src.transform.e - src.transform.b * src.transform.d) / 1_000_000.0
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)
            values, freqs = np.unique(arr, return_counts=True)
            for value, freq in zip(values, freqs):
                counts[int(value)] = counts.get(int(value), 0) + int(freq)
    return {
        "NonFlood_km2": counts.get(0, 0) * pixel_area_km2,
        "PermanentWater_km2": counts.get(1, 0) * pixel_area_km2,
        "Flood_km2": counts.get(2, 0) * pixel_area_km2,
        "NoData_pixels": counts.get(255, 0),
    }


def compare_maps(otsu_path: Path, lgbm_path: Path):
    import rasterio

    counts = {
        "both_nonflood": 0,
        "permanent_water": 0,
        "both_flood": 0,
        "otsu_only_flood": 0,
        "lightgbm_only_flood": 0,
        "common_valid": 0,
    }

    with rasterio.open(otsu_path) as osrc, rasterio.open(lgbm_path) as lsrc:
        if (osrc.width, osrc.height, osrc.crs) != (lsrc.width, lsrc.height, lsrc.crs) or not _same_transform(osrc.transform, lsrc.transform):
            raise RuntimeError("Otsu and LightGBM maps do not share the same raster grid.")
        pixel_area_km2 = abs(osrc.transform.a * osrc.transform.e - osrc.transform.b * osrc.transform.d) / 1_000_000.0

        for _, window in osrc.block_windows(1):
            o = osrc.read(1, window=window)
            l = lsrc.read(1, window=window)
            valid = (o != 255) & (l != 255)
            counts["common_valid"] += int(valid.sum())
            counts["both_nonflood"] += int((valid & (o == 0) & (l == 0)).sum())
            counts["permanent_water"] += int((valid & (o == 1) & (l == 1)).sum())
            counts["both_flood"] += int((valid & (o == 2) & (l == 2)).sum())
            counts["otsu_only_flood"] += int((valid & (o == 2) & (l != 2)).sum())
            counts["lightgbm_only_flood"] += int((valid & (o != 2) & (l == 2)).sum())

    intersection = counts["both_flood"]
    union = intersection + counts["otsu_only_flood"] + counts["lightgbm_only_flood"]
    otsu_flood = intersection + counts["otsu_only_flood"]
    lgbm_flood = intersection + counts["lightgbm_only_flood"]
    flood_iou = intersection / union if union else np.nan
    dice = (2 * intersection) / (otsu_flood + lgbm_flood) if (otsu_flood + lgbm_flood) else np.nan
    overall_same = counts["both_nonflood"] + counts["permanent_water"] + counts["both_flood"]
    overall_agreement = overall_same / counts["common_valid"] if counts["common_valid"] else np.nan

    return {
        **counts,
        "overall_agreement": overall_agreement,
        "flood_overlap_iou": flood_iou,
        "flood_dice": dice,
        "otsu_only_flood_km2": counts["otsu_only_flood"] * pixel_area_km2,
        "lightgbm_only_flood_km2": counts["lightgbm_only_flood"] * pixel_area_km2,
    }


if all(OTSU_MAPS[y].exists() and LGBM_MAPS[y].exists() for y in YEARS):
    area_rows = []
    spatial_rows = []
    for year in YEARS:
        o_area = map_area_statistics(OTSU_MAPS[year])
        l_area = map_area_statistics(LGBM_MAPS[year])
        area_rows.extend([
            {"Year": year, "Method": "Otsu", **o_area},
            {"Year": year, "Method": "LightGBM", **l_area},
        ])
        spatial_rows.append({"Year": year, **compare_maps(OTSU_MAPS[year], LGBM_MAPS[year])})

    area_table = pd.DataFrame(area_rows)
    spatial_table = pd.DataFrame(spatial_rows)
    area_table.to_csv(RESULTS_DIR / "flood_area_statistics.csv", index=False)
    spatial_table.to_csv(RESULTS_DIR / "spatial_agreement.csv", index=False)
    print(spatial_table[["Year", "overall_agreement", "flood_overlap_iou", "flood_dice", "lightgbm_only_flood_km2"]])

    if VERIFY_PUBLISHED_RESULTS:
        value_2024 = float(spatial_table.loc[spatial_table["Year"] == 2024, "lightgbm_only_flood_km2"].iloc[0])
        if not np.isclose(value_2024, 191.92, atol=0.15):
            raise RuntimeError(f"2024 LightGBM-only flood area changed: {value_2024:.2f} km2.")
else:
    print("Spatial map comparison skipped because final three-class maps are unavailable.")


## 11. Held-out TreeSHAP interpretation

Each fold-specific LightGBM explains only the event excluded from its development. Native LightGBM contribution values are calculated in raw-score space. The final column returned by `pred_contrib=True` is the expected-value term and is excluded from feature-importance ranking. No estimator is refitted specifically for interpretation.

In [ ]:
# =============================================================================
# 11. HELD-OUT TREESHAP INTERPRETATION
# =============================================================================

shap_rows = []
importance_table = None

if RUN_SHAP:
    for year in YEARS:
        fold = f"LOEO_test_{year}"
        test = loeo[(loeo["fold"] == fold) & (loeo["split"] == "test")].copy().reset_index(drop=True)
        model = fold_models[year]
        X = test[FEATURES].to_numpy(np.float32)

        contributions = model.booster_.predict(X, pred_contrib=True)
        if contributions.shape[1] != len(FEATURES) + 1:
            raise RuntimeError("Unexpected LightGBM contribution matrix shape.")

        values = contributions[:, :-1]
        expected = contributions[:, -1]
        raw_score = model.booster_.predict(X, raw_score=True)
        reconstructed = expected + values.sum(axis=1)
        if not np.allclose(raw_score, reconstructed, atol=1e-6, rtol=1e-6):
            raise RuntimeError(f"{year}: TreeSHAP contributions do not reconstruct the raw model score.")

        part = test[["year", "class_id", "global_pixel_row", "global_pixel_col"]].copy()
        for j, feature in enumerate(FEATURES):
            part[feature] = X[:, j]
            part[f"SHAP_{feature}"] = values[:, j]
        shap_rows.append(part)

    shap_data = pd.concat(shap_rows, ignore_index=True)
    mean_abs = np.mean(np.abs(shap_data[[f"SHAP_{f}" for f in FEATURES]].to_numpy(float)), axis=0)
    importance_table = pd.DataFrame({"Predictor": FEATURES, "Mean_abs_SHAP": mean_abs})
    importance_table["Relative_importance_percent"] = 100 * importance_table["Mean_abs_SHAP"] / importance_table["Mean_abs_SHAP"].sum()
    importance_table = importance_table.sort_values("Mean_abs_SHAP", ascending=False).reset_index(drop=True)
    importance_table["Rank"] = np.arange(1, len(importance_table) + 1)
    importance_table = importance_table[["Rank", "Predictor", "Mean_abs_SHAP", "Relative_importance_percent"]]

    shap_data.to_csv(RESULTS_DIR / "heldout_treeshap_values.csv", index=False)
    importance_table.to_csv(RESULTS_DIR / "heldout_treeshap_importance.csv", index=False)
    print(importance_table)

    if VERIFY_PUBLISHED_RESULTS:
        expected_top = ["VH_event", "Elevation", "VV_event"]
        observed_top = importance_table.head(3)["Predictor"].tolist()
        if observed_top != expected_top:
            raise RuntimeError(f"Held-out SHAP top three changed: {observed_top}")
        expected_pct = {"VH_event": 36.1, "Elevation": 17.6, "VV_event": 12.9}
        for feature, pct in expected_pct.items():
            observed = float(importance_table.loc[importance_table["Predictor"] == feature, "Relative_importance_percent"].iloc[0])
            if not np.isclose(observed, pct, atol=0.5):
                raise RuntimeError(f"{feature}: SHAP relative importance changed to {observed:.2f}%.")

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6.2, 5.0))
        d = importance_table.sort_values("Mean_abs_SHAP", ascending=True)
        ax.barh(d["Predictor"], d["Mean_abs_SHAP"])
        ax.set_xlabel("Mean |SHAP| (raw-score space)")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "heldout_treeshap_importance.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
    except Exception as exc:
        print("SHAP importance figure skipped:", exc)


## 12. Post-selection sensitivity analyses

These analyses are diagnostic and do not redefine the primary classifier. They include: (1) five replicate search seeds with 13 candidates per model family (195 evaluations), (2) fold-specific validation threshold sensitivity against the frozen common threshold of 0.53, (3) five predictor-group ablations evaluated at 0.53, and (4) prevalence-adjusted PPV/NPV calculated from the original held-out sensitivity and specificity.

In [ ]:
# =============================================================================
# 12. POST-SELECTION SENSITIVITY ANALYSES
# =============================================================================

REPLICATE_SEEDS = [101, 202, 303, 404, 505]
N_DRAWS = 12


def build_robustness_model(family, params):
    # This reproduces the post-selection sensitivity implementation used in the
    # archived analysis; the balanced reference design is retained unchanged.
    n_jobs = min(2, os.cpu_count() or 2)
    if family == "RandomForest":
        return RandomForestClassifier(**params, random_state=MODEL_SEED, n_jobs=n_jobs)
    if family == "XGBoost":
        return XGBClassifier(
            **params, objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=MODEL_SEED, n_jobs=n_jobs, verbosity=0
        )
    if family == "LightGBM":
        return LGBMClassifier(
            **params, objective="binary", random_state=MODEL_SEED,
            n_jobs=n_jobs, verbosity=-1
        )
    raise ValueError(family)


def candidate_validation_iou(family, params):
    values = []
    for fold in FOLDS:
        train = loeo[(loeo["fold"] == fold) & (loeo["split"] == "train")]
        val = loeo[(loeo["fold"] == fold) & (loeo["split"] == "validation")]
        model = build_robustness_model(family, dict(params))
        model.fit(train[FEATURES].to_numpy(float), train["class_id"].to_numpy(int))
        pred = (model.predict_proba(val[FEATURES].to_numpy(float))[:, 1] >= 0.50).astype(int)
        values.append(jaccard_score(val["class_id"], pred, pos_label=1, zero_division=0))
        del model
        gc.collect()
    return float(np.mean(values)), float(np.std(values, ddof=1))


if RUN_SENSITIVITY:
    # A. Seed/hyperparameter sensitivity: 5 x 13 x 3 = 195 evaluations.
    robustness_rows = []
    for replicate_seed in REPLICATE_SEEDS:
        for family in ("RandomForest", "XGBoost", "LightGBM"):
            candidates = [BASELINES[family], *list(ParameterSampler(SPACES[family], n_iter=N_DRAWS, random_state=replicate_seed))]
            for candidate_id, params in enumerate(candidates):
                mean_iou, sd_iou = candidate_validation_iou(family, params)
                robustness_rows.append({
                    "Replicate_Seed": replicate_seed,
                    "Model": family,
                    "Candidate_ID": candidate_id,
                    "Mean_Val_IoU": mean_iou,
                    "SD_Val_IoU": sd_iou,
                    "Parameters": json.dumps(dict(params), sort_keys=True, default=str),
                })

    robustness = pd.DataFrame(robustness_rows)
    robustness.to_csv(RESULTS_DIR / "hyperparameter_seed_sensitivity_all_candidates.csv", index=False)
    best_per_model_seed = (
        robustness.sort_values(["Replicate_Seed", "Model", "Mean_Val_IoU"], ascending=[True, True, False])
        .groupby(["Replicate_Seed", "Model"], as_index=False).first()
    )
    winner_per_seed = (
        best_per_model_seed.sort_values(["Replicate_Seed", "Mean_Val_IoU"], ascending=[True, False])
        .groupby("Replicate_Seed", as_index=False).first()
    )
    winner_per_seed.to_csv(RESULTS_DIR / "hyperparameter_seed_sensitivity_winners.csv", index=False)
    print("Robustness evaluations:", len(robustness))
    print("Winner counts:\n", winner_per_seed["Model"].value_counts())

    if VERIFY_PUBLISHED_RESULTS:
        counts = winner_per_seed["Model"].value_counts().to_dict()
        if counts.get("LightGBM", 0) != 4 or counts.get("XGBoost", 0) != 1:
            raise RuntimeError(f"Seed-level winner counts changed: {counts}")

    # B. Fold-specific threshold sensitivity.
    threshold_sensitivity_rows = []
    for year in YEARS:
        fold = f"LOEO_test_{year}"
        train = loeo[(loeo["fold"] == fold) & (loeo["split"] == "train")]
        val = loeo[(loeo["fold"] == fold) & (loeo["split"] == "validation")]
        test = loeo[(loeo["fold"] == fold) & (loeo["split"] == "test")]

        dev_model = build_locked_lgbm()
        dev_model.fit(train[FEATURES].to_numpy(float), train["class_id"].to_numpy(int))
        val_prob = dev_model.predict_proba(val[FEATURES].to_numpy(float))[:, 1]
        y_val = val["class_id"].to_numpy(int)

        scan = []
        for threshold in THRESHOLD_GRID:
            pred = (val_prob >= threshold).astype(int)
            scan.append((threshold, jaccard_score(y_val, pred, pos_label=1, zero_division=0), f1_score(y_val, pred, zero_division=0)))
        scan.sort(key=lambda x: (x[1], x[2]), reverse=True)
        fold_threshold = float(scan[0][0])

        final_model = build_locked_lgbm()
        fit = pd.concat([train, val], ignore_index=True)
        final_model.fit(fit[FEATURES].to_numpy(float), fit["class_id"].to_numpy(int))
        p_test = final_model.predict_proba(test[FEATURES].to_numpy(float))[:, 1]
        y_test = test["class_id"].to_numpy(int)
        iou_common = jaccard_score(y_test, (p_test >= LGBM_THRESHOLD).astype(int), pos_label=1, zero_division=0)
        iou_fold = jaccard_score(y_test, (p_test >= fold_threshold).astype(int), pos_label=1, zero_division=0)

        threshold_sensitivity_rows.append({
            "Year": year,
            "Validation_optimal_threshold": fold_threshold,
            "Validation_IoU": scan[0][1],
            "Validation_F1": scan[0][2],
            "HeldOut_IoU_common_0.53": iou_common,
            "HeldOut_IoU_fold_specific": iou_fold,
            "Delta_IoU_fold_minus_common": iou_fold - iou_common,
        })

    threshold_sensitivity = pd.DataFrame(threshold_sensitivity_rows)
    threshold_sensitivity.to_csv(RESULTS_DIR / "threshold_sensitivity.csv", index=False)
    print(threshold_sensitivity)

    # C. Predictor-group ablation at the frozen 0.53 threshold.
    ablations = {
        "Event_SAR_only": ["VV_event", "VH_event"],
        "Event_SAR_plus_Change": ["VV_event", "VH_event", "dVV", "dVH"],
        "Event_SAR_plus_TerrainHydro": ["VV_event", "VH_event", "Elevation", "Slope", "HAND", "Log_UPA"],
        "All_SAR_8": DYNAMIC_FEATURES,
        "Full_12": FEATURES,
    }
    ablation_rows = []
    for name, features in ablations.items():
        for year in YEARS:
            fold = f"LOEO_test_{year}"
            train = loeo[(loeo["fold"] == fold) & (loeo["split"] == "train")]
            test = loeo[(loeo["fold"] == fold) & (loeo["split"] == "test")]
            model = build_locked_lgbm()
            model.fit(train[features].to_numpy(float), train["class_id"].to_numpy(int))
            probability = model.predict_proba(test[features].to_numpy(float))[:, 1]
            prediction = (probability >= LGBM_THRESHOLD).astype(int)
            metric = binary_metrics(test["class_id"].to_numpy(int), prediction)
            ablation_rows.append({
                "Feature_Set": name,
                "N_Features": len(features),
                "Year": year,
                "Threshold": LGBM_THRESHOLD,
                **metric,
            })
            del model
            gc.collect()

    ablation = pd.DataFrame(ablation_rows)
    ablation_summary = (
        ablation.groupby(["Feature_Set", "N_Features"], as_index=False)
        .agg(
            Mean_HeldOut_IoU=("IoU", "mean"),
            SD_HeldOut_IoU=("IoU", "std"),
            Mean_F1=("F1", "mean"),
            Mean_Precision=("Precision", "mean"),
            Mean_Recall=("Recall", "mean"),
        )
        .sort_values("Mean_HeldOut_IoU", ascending=False)
    )
    ablation.to_csv(RESULTS_DIR / "predictor_ablation_event_results.csv", index=False)
    ablation_summary.to_csv(RESULTS_DIR / "predictor_ablation_summary.csv", index=False)
    print(ablation_summary)

    # D. Prevalence-adjusted PPV/NPV from original M3 sensitivity/specificity.
    prevalence_rows = []
    for _, row in heldout_metrics.iterrows():
        sensitivity = float(row["Recall"])
        specificity = float(row["Specificity"])
        for prevalence in (0.05, 0.10, 0.15):
            ppv = (sensitivity * prevalence) / (
                sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
            )
            npv = (specificity * (1 - prevalence)) / (
                specificity * (1 - prevalence) + (1 - sensitivity) * prevalence
            )
            prevalence_rows.append({
                "Year": int(row["Year"]),
                "Assumed_flood_prevalence": prevalence,
                "Expected_PPV": ppv,
                "Expected_NPV": npv,
            })
    prevalence_table = pd.DataFrame(prevalence_rows)
    prevalence_table.to_csv(RESULTS_DIR / "prevalence_adjusted_performance.csv", index=False)
    print(prevalence_table)


## 13. External NEMA flooded-location corroboration

This stage is optional in the public repository because the NEMA coordinates may be subject to redistribution restrictions. If `data/restricted/nema_flood_locations.csv` is present, the code samples the final Otsu and LightGBM products. Because the external dataset contains flooded locations only, the reported quantity is the proportion detected as flooded, with exact Clopper–Pearson 95% confidence intervals; it is not a complete accuracy assessment.

In [ ]:
# =============================================================================
# 13. EXTERNAL NEMA FLOODED-LOCATION CORROBORATION
# =============================================================================


def clopper_pearson(successes, total, alpha=0.05):
    if total == 0:
        return np.nan, np.nan
    lower = 0.0 if successes == 0 else beta.ppf(alpha / 2, successes, total - successes + 1)
    upper = 1.0 if successes == total else beta.ppf(1 - alpha / 2, successes + 1, total - successes)
    return float(lower), float(upper)


def sample_points_from_threeclass(path: Path, points_gdf):
    import rasterio

    with rasterio.open(path) as src:
        projected = points_gdf.to_crs(src.crs)
        coords = [(geom.x, geom.y) for geom in projected.geometry]
        values = np.fromiter((v[0] for v in src.sample(coords, indexes=1)), dtype=np.int16, count=len(coords))
        valid = values != 255
        flooded = np.where(valid, values == 2, False)
        return values, valid, flooded


if RUN_EXTERNAL_VALIDATION:
    if not NEMA_FILE.exists():
        print("NEMA external validation skipped: restricted coordinate file is not present.")
    elif not all(OTSU_MAPS[y].exists() and LGBM_MAPS[y].exists() for y in YEARS):
        print("NEMA external validation skipped: final three-class maps are unavailable.")
    else:
        import geopandas as gpd
        from shapely.geometry import Point

        points = pd.read_csv(NEMA_FILE).dropna(subset=["Longitude", "Latitude"]).reset_index(drop=True)
        if len(points) != 30:
            print(f"Note: external file contains {len(points)} locations; the archived analysis used 30.")
        points["External_ID"] = [f"NEMA_{i:02d}" for i in range(1, len(points) + 1)]
        gdf = gpd.GeoDataFrame(
            points,
            geometry=[Point(x, y) for x, y in zip(points["Longitude"], points["Latitude"])],
            crs="EPSG:4326",
        )

        sampled = {}
        all_valid = np.ones(len(gdf), dtype=bool)
        for year in YEARS:
            sampled[year] = {}
            for method, path in (("Otsu", OTSU_MAPS[year]), ("LightGBM", LGBM_MAPS[year])):
                values, valid, flooded = sample_points_from_threeclass(path, gdf)
                sampled[year][method] = (values, valid, flooded)
                all_valid &= valid

        # One common support is used for every event and both methods.
        validation_rows = []
        for year in YEARS:
            for method in ("Otsu", "LightGBM"):
                values, _, flooded = sampled[year][method]
                for i in range(len(gdf)):
                    validation_rows.append({
                        "External_ID": gdf.iloc[i]["External_ID"],
                        "Longitude": float(gdf.iloc[i]["Longitude"]),
                        "Latitude": float(gdf.iloc[i]["Latitude"]),
                        "Year": year,
                        "Method": method,
                        "Common_Valid": bool(all_valid[i]),
                        "Raster_Class": int(values[i]),
                        "Flood_Detected": int(flooded[i]) if all_valid[i] else np.nan,
                    })
        external_validation = pd.DataFrame(validation_rows)
        external_validation.to_csv(RESULTS_DIR / "nema_external_validation.csv", index=False)

        summary_rows = []
        for (year, method), d in external_validation.dropna(subset=["Flood_Detected"]).groupby(["Year", "Method"]):
            total = len(d)
            detected = int(d["Flood_Detected"].sum())
            low, high = clopper_pearson(detected, total)
            summary_rows.append({
                "Year": int(year),
                "Method": method,
                "Valid_external_points": total,
                "Flood_points_detected": detected,
                "Detection_percent": 100 * detected / total,
                "CI95_Lower_percent": 100 * low,
                "CI95_Upper_percent": 100 * high,
            })
        external_summary = pd.DataFrame(summary_rows)
        external_summary.to_csv(RESULTS_DIR / "nema_external_validation_summary.csv", index=False)
        print("Common valid external locations:", int(all_valid.sum()))
        print(external_summary)

        if VERIFY_PUBLISHED_RESULTS:
            expected = {
                (2018, "LightGBM"): (16, 21),
                (2018, "Otsu"): (15, 21),
                (2022, "LightGBM"): (19, 21),
                (2022, "Otsu"): (19, 21),
                (2024, "LightGBM"): (20, 21),
                (2024, "Otsu"): (20, 21),
            }
            for key, (success, total) in expected.items():
                row = external_summary[(external_summary["Year"] == key[0]) & (external_summary["Method"] == key[1])]
                if row.empty:
                    raise RuntimeError(f"Missing NEMA summary row: {key}")
                obs = (int(row.iloc[0]["Flood_points_detected"]), int(row.iloc[0]["Valid_external_points"]))
                if obs != (success, total):
                    raise RuntimeError(f"NEMA result changed for {key}: {obs} != {(success, total)}")


## 14. Reproducibility audit and manuscript outputs

The final cell records software versions, locked analytical constants, and SHA-256 hashes for files available in the repository. These hashes provide a compact audit trail without exposing private cloud-storage paths.

In [ ]:
# =============================================================================
# 14. REPRODUCIBILITY AUDIT AND MANUSCRIPT OUTPUTS
# =============================================================================


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def package_version(name):
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return None


files_to_hash = []
for path in [LOEO_FILE, REFERENCE_PIXELS_FILE, GRID_FILE, JRC_PERMANENT_WATER, *PREDICTOR_STACKS.values(), *OTSU_MAPS.values(), *LGBM_MAPS.values()]:
    if path.exists():
        files_to_hash.append(path)
for path in sorted(RESULTS_DIR.glob("*")):
    if path.is_file():
        files_to_hash.append(path)

manifest = {
    "study_years": list(YEARS),
    "analysis_crs": ANALYSIS_CRS,
    "pixel_size_m": PIXEL_SIZE_M,
    "predictor_order": FEATURES,
    "lightgbm_threshold": LGBM_THRESHOLD,
    "otsu_histogram": {
        "minimum_db": OTSU_HIST_MIN_DB,
        "maximum_db": OTSU_HIST_MAX_DB,
        "bin_width_db": OTSU_BIN_WIDTH_DB,
    },
    "expected_otsu_thresholds_db": EXPECTED_OTSU_THRESHOLDS,
    "model_seed": MODEL_SEED,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_replicates": N_BOOTSTRAP,
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in [
            "numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm",
            "joblib", "rasterio", "geopandas", "earthengine-api", "matplotlib"
        ]
    },
    "file_sha256": {
        str(path.relative_to(REPO_ROOT) if REPO_ROOT in path.parents else path): sha256_file(path)
        for path in files_to_hash
    },
}

manifest_path = RESULTS_DIR / "reproducibility_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("Reproducibility manifest:", manifest_path)
print("Files hashed:", len(manifest["file_sha256"]))
print("Workflow complete.")
